### Installing NuGet Packages

- Microsoft.Data.Analysis - To load the processed data as data frame and work with it.
- Plotly.NET - To visualize data distribution and visually verify the closeness of synthetic data.
- Microsoft.ML - To load pre-trained model and make predictions

In [33]:
#r "nuget: Microsoft.Data.Analysis, 0.23.0"
#r "nuget: Microsoft.ML, 5.0.0"
#r "nuget: Plotly.NET, 6.0.0-preview.1"
#r "nuget: Plotly.NET.CSharp, 0.14.0-preview.1"
#r "nuget: Plotly.NET.Interactive, 6.0.0-preview.1"

Installed Packages Microsoft.Data.Analysis, 0.23.0 Microsoft.ML, 5.0.0 Plotly.NET, 6.0.0-preview.1 Plotly.NET.CSharp, 0.14.0-preview.1 Plotly.NET.Interactive, 6.0.0-preview.1

### Loading Data

1. Read CSV data as `DataFrame`.
2. Delete non-synthetic columns - `status` (predicted by model) and `ltv` (derived from `loan_amount` and `property_value`).

In [34]:
using Microsoft.Data.Analysis;

DataFrame dataFrame = DataFrame.LoadCsv("./processed-data.csv");


int totalDefaulted = dataFrame["status"].Cast<bool>()
    .Where(status => status).Count();

Console.WriteLine($@"
Loan Default Rate: {Math.Round(100f * totalDefaulted/dataFrame.Rows.Count, 2)}%
");

// we don't want to generate data for status and ltv (loan_amount / property_value)
string[] skippedColumns = ["status", "ltv"];

foreach( var columnName in skippedColumns)
{
    dataFrame.Columns.Remove(columnName);
}

dataFrame.Head(5).Display();


Loan Default Rate: 24.64%



index,loan_amount,rate_of_interest,upfront_charges,term,property_value,income,credit_score,age,region,dtir1
0,116500,3.99,0,360,118000,1740,758,25-34,south,45
1,206500,3.99,0,360,308000,4980,552,55-64,North,37
2,406500,4.56,595,360,508000,9480,834,35-44,south,46
3,456500,4.25,0,360,658000,11880,587,45-54,North,42
4,696500,4,0,360,758000,10440,602,25-34,North,39


### Convert `age` and `region` to Numeric

Since we are generating synthetic data using cumulative distribution functions (CDF), it is useful to convert categorical columns (`age` and `region`) to numeric. We do this using a simple one-to-one mapping, and will use the inverse mapping to get actual feature value for prediction.

In [ ]:
// convert age and region columns to numeric for generation
// we should remember to convert back while making predictions

// see the distinct values for age and region
dataFrame["age"].Cast<string>().Distinct().Display();
dataFrame["region"].Cast<string>().Distinct().Display();

[ 25-34, 55-64, 35-44, 45-54, 65-74, >74, <25, unknown ]

[ south, North, central, North-East ]

In [36]:
// for age we map to their group averages
var floatAges = dataFrame["age"].Cast<string>()
    .Select(age => age switch 
    {
        "<25" => 20f,
        "25-34" => 30f,
        "35-44" => 40f,
        "45-54" => 50f,
        "55-64" => 60f,
        "65-74" => 70f,
        ">74" => 80f,
        _ => 0f
    });

dataFrame["age"] = new SingleDataFrameColumn("", floatAges);

// for region we use arbitrary mapping
var floatRegions = dataFrame["region"].Cast<string>()
    .Select(region => region switch
    {
        "North-East" => 1f,
        "North" => 2f,
        "central" => 3f,
        "south" => 4f,
        _ => 0f
    });

dataFrame["region"] = new SingleDataFrameColumn("", floatRegions);

dataFrame.Head(10).Display();

index,loan_amount,rate_of_interest,upfront_charges,term,property_value,income,credit_score,age,region,dtir1
0,116500,3.99,0,360,118000,1740,758,30,4,45
1,206500,3.99,0,360,308000,4980,552,60,2,37
2,406500,4.56,595,360,508000,9480,834,40,4,46
3,456500,4.25,0,360,658000,11880,587,50,2,42
4,696500,4,0,360,758000,10440,602,30,2,39
5,706500,3.99,370,360,1008000,10080,864,40,2,40
6,346500,4.5,5120,360,438000,5040,860,60,2,44
7,266500,4.125,5609.88,360,308000,3780,863,60,2,42
8,376500,4.875,1150,360,478000,5580,580,60,3,44
9,436500,3.49,2316.5,360,688000,6720,788,60,4,30


### Plots for Source Data

The frequency distribution for all columns of the data frame are plotted below.

These plots give a visual idea of how the synthetic generator should simulate new data.

In [37]:
using Plotly.NET.CSharp;

foreach (var column in dataFrame.Columns)
{
    Chart.Histogram<float, string, string>(
        X: new(column.Cast<float>(), true),
        HistNorm: Plotly.NET.StyleParam.HistNorm.Percent
    )
    .WithXAxisStyle<float, string, string>(column.Name)
    .Display();
}

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

### Cumulative Distribution Functions

A CDF gives the probability that a random variable is less than or equal a value.

For example, if `cdf(25) = 0.875`, there is 87.5% chance that the "random variable" is less than 25.

1. CDF computation: `valueCounts` is built per column, cumulative probabilities are computed from the counts (running sum / total), and a `Probabilities` column is added to capture the CDF points.
2. Downsample & map: If there are many distinct values the code reduces rows to at most 20.
3. Persist & visualize: Each column's CDF is stored in the `cdfs` dictionary (keyed by column name), written out to `cdfs.json`.

In [38]:
using System.IO;
using System.Text.Json;

// cumulative distribution function

// x = [1,1,1,2,2,2,2,2,3,3,4,4,5,5,5,5]

// "valueCounts" data frame = frequency table
// index    x    f       cf (cumulative frequency)       probailities
// 0        1    3       3                                3/16
// 1        2    5       3 + 5 = 8                        1/2
// 2        3    2       8 + 2 = 10                       5/8
// 3        4    2       10 + 2 = 12                      3/4   
// 4        5    4       12 + 4 = 16                      1   

// x = Values, f = Counts

// cdf(4) = 75%
// The probability of the random variable "X" being less than or equal to 4 is 75% 

// "column_name" : {<x> : <probabilities>}
Dictionary<string, Dictionary<float, float>> cdfs = [];

foreach (var column in dataFrame.Columns)
{
    DataFrame valueCounts = column.ValueCounts().OrderBy("Values");

    long[] counts = [..valueCounts["Counts"].Cast<long>()];
    long totalCount = counts.Sum();

    float[] probabilities = [
        ..Enumerable.Range(0, counts.Length)
        .Select(index => 1f * counts[0..(index + 1)].Sum() / totalCount)
    ];

    valueCounts.Columns.Add(new PrimitiveDataFrameColumn<float>(
        name: "Probabilities", 
        probabilities
    ));

    // only keep 20 unique values per column
    int maxRows = 20;
    long numberOfRows = valueCounts.Rows.Count; // count of unique values

    // 5 -> [0, 1, 2, 3, 4] (step = 1 -> 5/20 + 1)
    // 28 -> [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 27] (step = 2 -> 28/20 + 1)

    if (numberOfRows > maxRows)
    {
        var step = Math.Ceiling(1f * numberOfRows / maxRows);
        var numberOfIndices = (int) Math.Ceiling(1f * numberOfRows / step);

        long[] indices = [
            ..Enumerable.Range(0, numberOfIndices)
                .Select(index =>  index * (int) step)
        ];

        // make sure that the last index is included
        indices[^1] = numberOfRows - 1;

        valueCounts = valueCounts.Filter(new PrimitiveDataFrameColumn<long>(
            "", indices
        ));
    }

    float[] cdfValues = [..valueCounts["Values"].Cast<float>()];
    float[] cdfProbabilities = [..valueCounts["Probabilities"].Cast<float>()];

    // empirical CDF for the column
    Dictionary<float, float> cdf = cdfValues
        .Zip(cdfProbabilities, (value, probability) => new {value, probability})
        .ToDictionary(kv => kv.value, kv => kv.probability);

    cdfs.Add(column.Name, cdf);

    // visualize CDF
    Chart.Line<float, float, string>(x: cdfValues, y: cdfProbabilities)
        .WithXAxisStyle<float, float, string>(column.Name)
        .WithYAxisStyle<float, float, string>("Cumulative Probability")
        .Display();
}

File.WriteAllText("./cdfs.json", JsonSerializer.Serialize(cdfs));

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

### Synthetic Data Generation
We can use inverse of cdf to sythetically generate data for a random variable, preserving the distribution of population.

1. Use the cdfs data to generate sythetic data.
2. Verify the distribution visually.
3. Use the model to predict status columns.

In [39]:
var random = new Random();

cdfs = JsonSerializer.Deserialize<Dictionary<string, Dictionary<float, float>>>(
    File.ReadAllText("./cdfs.json")
);

int numObservations = 1000;

// assumption: all the columns represent independent random variables
var syntheticData = new DataFrame(cdfs.Select(kv => new SingleDataFrameColumn(
    // kv.Key -> column_name
    // kv.Value -> <x> : <probability>
    kv.Key, Enumerable.Range(0, numObservations).Select(_ => {
        // generate CDF value (random value between 0 & 1)
        var probability = random.NextDouble();
        // find the "x" value that corresponds to the CDF value
        return kv.Value.First(kv => kv.Value > probability).Key;
    })
)));

// visually verify distribution
foreach (var column in syntheticData.Columns) 
{    
    Chart.Histogram<float, string, string>(
        X: new(column.Cast<float>(), true),
        HistNorm: Plotly.NET.StyleParam.HistNorm.Percent
    )
    .WithXAxisStyle<float, string, string>(column.Name)
    .Display();
}

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

<!-- Plotly chart will be drawn inside this DIV -->

In [40]:
// revert age and region columns
var stringAges = syntheticData["age"].Cast<float>()
    .Select(age => age switch 
    {
        0 => "unknown",
        < 25 => "<25",
        < 35 => "25-34",
        < 45 => "35-44",
        < 55 => "45-54",
        < 65 => "55-64",
        < 75 => "65-74",
        < 85 => ">74",
        _ => "unknown"
    });

syntheticData["age"] = new StringDataFrameColumn("", stringAges);

var stringRegions = syntheticData["region"].Cast<float>()
    .Select(region => region switch
    {
        1f => "North-East",
        2f => "North",
        3f => "central",
        4f => "south",
        _ => "unknown"
    });

syntheticData["region"] = new StringDataFrameColumn("", stringRegions);

// compute ltv
syntheticData["ltv"] = syntheticData["loan_amount"] / syntheticData["property_value"];

syntheticData.Head(5).Display();

index,loan_amount,rate_of_interest,upfront_charges,term,property_value,income,credit_score,age,region,dtir1,ltv
0,346500,3.3,4576.25,360,208000,9180,773,65-74,North,35,1.6658654
1,346500,4.5,0,360,808000,6120,626,45-54,central,38,0.42883664
2,126500,3.74,0,180,408000,6120,731,35-44,North,56,0.31004903
3,346500,3.9,3046.49,360,408000,15300,878,<25,south,41,0.8492647
4,456500,4.025,0,360,608000,6120,668,35-44,south,38,0.75082237


In [ ]:
using Microsoft.ML;

var mlContext = new MLContext();

var model = mlContext.Model.Load("./model.zip", out var inputSchema);

var gameplayData = model.Transform(syntheticData);

syntheticData["status"] = gameplayData.ToDataFrame(numObservations)["PredictedLabel"];

syntheticData.Head(5).Display();

int syntheticDefaulted = syntheticData["status"].Cast<bool>()
    .Where(status => status).Count();

Console.WriteLine($@"
Loan Default Rate: {Math.Round(100f * syntheticDefaulted/syntheticData.Rows.Count, 2)}%
");

index,loan_amount,rate_of_interest,upfront_charges,term,property_value,income,credit_score,age,region,dtir1,ltv,status
0,346500,3.3,4576.25,360,208000,9180,773,65-74,North,35,1.6658654,False
1,346500,4.5,0,360,808000,6120,626,45-54,central,38,0.42883664,False
2,126500,3.74,0,180,408000,6120,731,35-44,North,56,0.31004903,True
3,346500,3.9,3046.49,360,408000,15300,878,<25,south,41,0.8492647,False
4,456500,4.025,0,360,608000,6120,668,35-44,south,38,0.75082237,True



Loan Default Rate: 29.8%

